# Занятие 9. Генеративно-состязательные сети (GAN)

## Цели занятия:
- Изучить архитектуру GAN (Generator vs Discriminator)
- Реализовать DCGAN для генерации изображений
- Освоить техники стабилизации обучения GAN
- Сравнить качество генерации с VAE

---

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import random
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Шаг 1. Загрузка данных (FashionMNIST)

In [ ]:
# Normalize to [-1, 1] for Tanh activation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)

print(f"Train samples: {len(train_dataset)}")

---
## Шаг 2. Реализация Генератора

In [ ]:
class Generator(nn.Module):
    """DCGAN Generator.
    
    Uses ConvTranspose2d for upsampling.
    Input: Random noise z (latent_dim)
    Output: Generated image (1, 28, 28)
    """
    
    def __init__(self, latent_dim=100):
        super().__init__()
        
        self.model = nn.Sequential(
            # Input: (B, latent_dim, 1, 1)
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # (B, 256, 4, 4)
            
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # (B, 128, 8, 8)
            
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # (B, 64, 16, 16)
            
            nn.ConvTranspose2d(64, 1, 4, 2, 3, bias=False),  # Adjusted for 28x28
            nn.Tanh()
            # (B, 1, 28, 28)
        )
    
    def forward(self, z):
        z = z.view(-1, 100, 1, 1)
        return self.model(z)

# Test
G = Generator(latent_dim=100).to(device)
test_z = torch.randn(1, 100).to(device)
test_output = G(test_z)
print(f"Generator output shape: {test_output.shape}")

---
## Шаг 3. Реализация Дискриминатора

In [ ]:
class Discriminator(nn.Module):
    """DCGAN Discriminator.
    
    Uses strided Conv2d for downsampling.
    Input: Image (1, 28, 28)
    Output: Probability (real/fake)
    """
    
    def __init__(self):
        super().__init__()
        
        self.model = nn.Sequential(
            # Input: (B, 1, 28, 28)
            nn.Conv2d(1, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (B, 64, 14, 14)
            
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # (B, 128, 7, 7)
            
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # (B, 256, 3, 3)
            
            nn.Conv2d(256, 1, 3, 1, 0, bias=False),
            nn.Sigmoid()
            # (B, 1, 1, 1)
        )
    
    def forward(self, x):
        return self.model(x).view(-1, 1)

# Test
D = Discriminator().to(device)
test_img = torch.randn(1, 1, 28, 28).to(device)
test_output = D(test_img)
print(f"Discriminator output shape: {test_output.shape}")

---
## Шаг 4. Инициализация весов

In [ ]:
def weights_init(m):
    """Initialize weights for DCGAN.
    
    - Conv: N(0, 0.02)
    - BatchNorm: N(1, 0.02) for weight, 0 for bias
    """
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Apply initialization
G = Generator(latent_dim=100).apply(weights_init).to(device)
D = Discriminator().apply(weights_init).to(device)

print("Weights initialized!")

---
## Шаг 5. Функции потерь и оптимизаторы

In [ ]:
criterion = nn.BCELoss()

# Optimizers with momentum (beta1=0.5 is important for GAN stability)
opt_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Labels
real_label = 1.0
fake_label = 0.0

print("Optimizer setup complete!")

---
## Шаг 6. Цикл обучения GAN

In [ ]:
history = {'g_loss': [], 'd_loss': []}
fixed_noise = torch.randn(16, 100, 1, 1).to(device)  # For visualization

num_epochs = 20

print("="*50)
print("Training GAN")
print("="*50)

for epoch in range(num_epochs):
    for i, (images, _) in enumerate(train_loader):
        batch_size = images.size(0)
        images = images.to(device)
        
        # --- Train Discriminator ---
        D.zero_grad()
        
        # Real images
        real_labels = torch.full((batch_size,), real_label).to(device)
        output_real = D(images)
        d_loss_real = criterion(output_real, real_labels.unsqueeze(1))
        
        # Fake images
        noise = torch.randn(batch_size, 100, 1, 1).to(device)
        fake_images = G(noise)
        fake_labels = torch.full((batch_size,), fake_label).to(device)
        output_fake = D(fake_images.detach())
        d_loss_fake = criterion(output_fake, fake_labels.unsqueeze(1))
        
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        opt_D.step()
        
        # --- Train Generator ---
        G.zero_grad()
        
        noise = torch.randn(batch_size, 100, 1, 1).to(device)
        fake_images = G(noise)
        # Generator wants Discriminator to say "real"
        real_labels_g = torch.full((batch_size,), real_label).to(device)
        output = D(fake_images)
        g_loss = criterion(output, real_labels_g.unsqueeze(1))
        
        g_loss.backward()
        opt_G.step()
        
        history['g_loss'].append(g_loss.item())
        history['d_loss'].append(d_loss.item())
    
    print(f"Epoch {epoch+1}: G_loss={g_loss.item():.4f}, D_loss={d_loss.item():.4f}")

---
## Шаг 7. Визуализация прогресса генерации

In [ ]:
# Generate images
G.eval()
with torch.no_grad():
    noise = torch.randn(16, 100, 1, 1).to(device)
    generated = G(noise)

# Plot
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img = generated[i].cpu().squeeze()
    img = img * 0.5 + 0.5  # Denormalize from [-1,1] to [0,1]
    ax.imshow(img, cmap='gray')
    ax.axis('off')

plt.suptitle('Generated Images (Final Epoch)', fontsize=14)
plt.tight_layout()
plt.savefig('gan_generated.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 8. Графики функции потерь

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['g_loss'], label='Generator Loss', alpha=0.7)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Generator Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['d_loss'], label='Discriminator Loss', color='orange', alpha=0.7)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_title('Discriminator Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gan_loss.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 9. Сравнение VAE и GAN

In [ ]:
import pandas as pd

comparison = {
    'Aspect': ['Training Stability', 'Image Quality', 'Mode Coverage', 'Latent Space'],
    'VAE': ['Stable', 'Blurry', 'Good (covers all modes)', 'Continuous & Interpretable'],
    'GAN': ['Unstable (oscillating)', 'Sharp', 'Poor (mode collapse risk)', 'Not directly interpretable']
}

df = pd.DataFrame(comparison)
print("\n" + "="*60)
print("VAE vs GAN Comparison")
print("="*60)
print(df.to_string(index=False))

---
## Домашнее задание

1. Поэкспериментировать с размером латентного пространства (50, 100, 200)
2. Реализовать Conditional GAN (генерация по классу)
3. Попробовать WGAN-GP для улучшения стабильности обучения
4. Сравнить качество генерации на разных эпохах (5, 10, 20)

---
## Контрольные вопросы

1. Почему Generator хочет максимизировать ошибку Discriminator?
2. Что такое Mode Collapse и как его обнаружить?
3. Зачем нужна инициализация весов в GAN?
4. Почему GAN сложнее обучать чем VAE?